# Gas-turbine engine — Brayton cycle analysis & optimisation

**Author:** Esteban López · Industrial Engineer  
**Tools:** Brayton (gas-turbine) thermodynamics, cold-air-standard analysis, Python.

## Problem
Analyse the air-standard **Brayton cycle** that powers every gas turbine and **jet engine**: compute each state,
the thermal efficiency (ideal and with real compressor/turbine losses), the **back-work ratio**, and find the
**pressure ratio that maximises net work** for a given turbine-inlet temperature (TIT).

**Design point:** pressure ratio rp = 12, TIT = 1400 K, compressor η = 0.85, turbine η = 0.88, inlet 288 K / 100 kPa.

> Reproducible, cold-air-standard (cp = 1.005 kJ/kg·K, k = 1.4). `pip install numpy matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD, GREY = '#0b1f3a', '#16b3c7', '#f2c14e', '#5b6b7f'
cp, k, R = 1.005, 1.4, 0.287       # cold-air-standard
T1, P1, T3 = 288.0, 100.0, 1400.0  # inlet T/P, turbine-inlet T (TIT)
etac, etat = 0.85, 0.88

## 1. Cycle model
1 = compressor inlet, 2 = compressor exit, 3 = turbine inlet (after combustor), 4 = turbine exit.

In [ ]:
def brayton(rp, T1=T1, T3=T3, etac=etac, etat=etat):
    pr = rp ** ((k - 1) / k)
    T2s = T1 * pr;  T2 = T1 + (T2s - T1) / etac      # real compression
    T4s = T3 / pr;  T4 = T3 - etat * (T3 - T4s)      # real expansion
    wc = cp * (T2 - T1); wt = cp * (T3 - T4)
    wnet = wt - wc; qin = cp * (T3 - T2)
    return dict(T2=T2, T4=T4, wc=wc, wt=wt, wnet=wnet, qin=qin, eff=wnet/qin, bwr=wc/wt)

r = brayton(12); ideal = brayton(12, etac=1, etat=1)
print(f"Real  : eff = {r['eff']*100:.1f} %   wnet = {r['wnet']:.0f} kJ/kg   BWR = {r['bwr']*100:.0f} %")
print(f"Ideal : eff = {ideal['eff']*100:.1f} %   wnet = {ideal['wnet']:.0f} kJ/kg")
print(f"Compressor work {r['wc']:.0f} | Turbine work {r['wt']:.0f} | q_in {r['qin']:.0f} kJ/kg | T2={r['T2']:.0f}K T4={r['T4']:.0f}K")

## 2. T–s diagram (two isobars)

In [ ]:
def s(T, P): return cp*np.log(T/T1) - R*np.log(P/P1)
P2 = 12*P1; T2, T4 = r['T2'], r['T4']
fig, ax = plt.subplots(figsize=(8.5,5.5))
Tlo = np.linspace(280, 820, 80);  ax.plot(s(Tlo,P1), Tlo, color=GREY, ls='--', lw=1.2)
Thi = np.linspace(560, 1450, 80); ax.plot(s(Thi,P2), Thi, color=GREY, ls='--', lw=1.2)
pts = [(s(T1,P1),T1),(s(T2,P2),T2),(s(T3,P2),T3),(s(T4,P1),T4),(s(T1,P1),T1)]
ax.plot([p[0] for p in pts],[p[1] for p in pts], color=CYAN, lw=2.4)
for (x,T),l in zip(pts[:4],'1234'):
    ax.plot(x,T,'o',color=GOLD); ax.text(x,T+25,l,fontweight='bold',ha='center')
ax.set_xlabel('Δs (kJ/kg·K)'); ax.set_ylabel('T (K)')
ax.set_title('Brayton cycle T–s diagram', fontweight='bold', color=NAVY); plt.tight_layout(); plt.show()

## 3. Optimum pressure ratio
Unlike efficiency (which keeps rising with rp for the ideal cycle), **net work has a clear maximum**. With real
component efficiencies the optimum shifts and the choice becomes a work-vs-efficiency trade-off.

In [ ]:
rps = np.linspace(2, 40, 100)
wn = [brayton(x)['wnet'] for x in rps]; ef = [brayton(x)['eff']*100 for x in rps]
rp_opt = rps[int(np.argmax(wn))]
fig, ax1 = plt.subplots(figsize=(8.6,5))
ax1.plot(rps, wn, color=CYAN, lw=2.4, label='net work'); ax1.axvline(rp_opt, color=GREY, ls=':')
ax1.set_xlabel('Pressure ratio rp'); ax1.set_ylabel('Net work (kJ/kg)', color=CYAN)
ax2 = ax1.twinx(); ax2.plot(rps, ef, color=GOLD, lw=2.2); ax2.set_ylabel('Efficiency (%)', color='#b8902a')
ax1.set_title(f'Net work peaks at rp ≈ {rp_opt:.0f}', fontweight='bold', color=NAVY)
ax1.grid(alpha=.2); plt.tight_layout(); plt.show()
print(f'Optimum pressure ratio for max net work: rp ≈ {rp_opt:.1f}')

## 4. The back-work ratio — and the aerospace link
The compressor consumes **~56 %** of the turbine's output (vs **<1 %** for the pump in a steam Rankine cycle). That
is why a gas turbine lives or dies by its compressor and turbine efficiencies — a few points of η move the whole cycle.

**Aerospace connection:** a **turbojet is an open Brayton cycle**. The turbine only drives the compressor; the
remaining energy accelerates the exhaust to produce **thrust** (F = ṁ(V_exit − V_inlet)). The same analysis sizes
aircraft engines, auxiliary power units (APUs) and the gas turbines behind a lot of ground and marine power — a
direct bridge from industrial energy systems to propulsion and aerospace.

In [ ]:
for TIT in [1200, 1400, 1600]:
    rr = brayton(12, T3=TIT)
    print(f'TIT {TIT} K  ->  eff {rr["eff"]*100:4.1f} %   net work {rr["wnet"]:4.0f} kJ/kg   BWR {rr["bwr"]*100:.0f} %')
print('\nHigher TIT is the single biggest lever — and why turbine-blade materials & cooling are the frontier.')

## 5. Conclusion

- The rp = 12 / TIT = 1400 K gas turbine reaches **≈ 51 % ideal** and **≈ 36 % real** thermal efficiency; the gap is driven by the **high back-work ratio (~56 %)**.
- **Net work has an optimum pressure ratio** (here rp ≈ 10–12) — a real design choice, unlike the monotonic ideal-efficiency curve.
- **Turbine-inlet temperature is the dominant lever**, which is why materials and blade cooling define modern engines.
- The same cycle, run open, is the **turbojet** — linking this analysis straight to jet propulsion and aerospace power.

*Natural extensions: regeneration, intercooling and reheat (to cut the back-work penalty), and a combined Brayton-Rankine cycle — exactly how the most efficient power plants on Earth work.*